
#### Spark SQL Basics
  - spark.sql() API
  - Temporary Views 
  - Global Temporary Views
  - Permanent Views
  - DataFrame <-> SQL interoperability
  - Schema & explain()


In [0]:

# ---------------------------------------------------------------------------
# CREATING DataFrames (foundation for all SQL operations)
# ---------------------------------------------------------------------------
employees = [
    (1, "Alice",  "Engineering", 95000, "2020-03-15"),
    (2, "Bob",    "Marketing",   72000, "2019-07-01"),
    (3, "Carol",  "Engineering", 110000,"2018-11-20"),
    (4, "Dave",   "HR",          60000, "2021-01-10"),
    (5, "Eve",    "Marketing",   85000, "2020-09-25"),
    (6, "Frank",  "Engineering", 98000, "2017-05-30"),
    (7, "Grace",  "HR",          63000, "2022-02-14"),
    (8, "Hank",   None,          70000, "2021-08-08"),   # NULL dept
]
emp_schema = "id INT, name STRING, department STRING, salary INT, hire_date STRING"
emp_df = spark.createDataFrame(employees, schema=emp_schema)

departments = [
    ("Engineering", "New York"),
    ("Marketing",   "Chicago"),
    ("HR",          "Dallas"),
    ("Finance",     "Boston"),   # no employees in this dept
]
dept_schema = "department STRING, location STRING"
dept_df = spark.createDataFrame(departments, schema=dept_schema)

##### Temporary Views

- createTempView         → raises AnalysisException if name already exists
- createOrReplaceTempView→ replaces silently (preferred)

In [0]:
emp_df.createOrReplaceTempView("employees")
dept_df.createOrReplaceTempView("departments")

# SQL
spark.sql("Select * from employees where department = 'Engineering'").show()


##### GLOBAL TEMPORARY VIEW  

- cross-session — lives in `global_temp` database
- Scope: all SparkSessions in the same application (same JVM)
- Access: ALWAYS prefix with  global_temp.<view_name>

In [0]:
emp_df.createOrReplaceGlobalTempView("employees")

spark.sql("Select * from global_temp.employees where department = 'Engineering'").show()

##### PERMANENT VIEW 

- persisted in metastore — survives restarts - df.write.saveAsTable("<TblName>")
- Requires a database context (default = 'default')
- In Databricks: stored in Unity Catalog or Hive Metastore

In [0]:
emp_df.write.mode("overwrite").saveAsTable("employeeTable")

spark.sql("drop view if exists vw_high_earners")
spark.sql("""Create view vw_high_earners AS
          Select name, department from default.employeeTable
          where salary > 100000
          """)

spark.sql("select * from vw_high_earners").show()

In [0]:
spark.sql("select * from default.employeetable").show()

In [0]:
spark.sql("show views").show()

##### DataFrame API  <-->  SQL EQUIVALENCE  

- Both produce identical execution plans — Catalyst treats them the same


In [0]:
print("=== DataFrame API vs SQL — same result ===")

# SQL style
sql_result = spark.sql("""
    SELECT department, COUNT(*) AS headcount, AVG(salary) AS avg_salary
    FROM employees
    WHERE department IS NOT NULL
    GROUP BY department
    ORDER BY avg_salary DESC
""")


In [0]:
from pyspark.sql.functions import col

df_result = emp_df.filter(emp_df.department.isNotNull()) \
      .groupBy("department") \
      .agg({"salary":"avg", "id": "count"}) \
      .withColumnRenamed("avg(salary)", "AverageSalary") \
      .withColumnRenamed("count(id)", "Count") \
      .orderBy(col("AverageSalary").desc())

In [0]:
sql_result.show()

In [0]:
df_result.show()

In [0]:
# ---------------------------------------------------------------------------
# EXPLAIN — viewing the execution plan (exam: know all modes)
#    modes: simple (default) | extended | codegen | cost | formatted
# ---------------------------------------------------------------------------
query = spark.sql("""
    SELECT e.name, e.salary, d.location
    FROM employees e
    JOIN departments d ON e.department = d.department
    WHERE e.salary > 80000
""")

print("===  explain() — simple (physical plan only) ===")
query.explain()

print("===  explain('extended') — parsed/analyzed/optimized/physical ===")
query.explain("extended")

print("===  explain('formatted') — cleaner readable format ===")
query.explain("formatted")

In [0]:
query.explain("codegen")

In [0]:

# ---------------------------------------------------------------------------
# SCHEMA inspection
# ---------------------------------------------------------------------------
print("=== Schema ===")
emp_df.printSchema()
print("StructType:", emp_df.schema)
print("Column names:", emp_df.columns)
print("dtypes:", emp_df.dtypes)

In [0]:
spark.stop()